---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# 10. Full Comparison — All Methods

## Objective

Compare **all methods** tested for Sonic Log (DT) prediction:
- **Machine Learning (tree-based):** XGBoost, HistGB, LightGBM, RandomForest
- **Neural Networks:** MLP, 1D-CNN

Validation: **LOWO (Leave-One-Well-Out)** — 27 wells from the Sergipe-Alagoas Basin.

**Reference algorithm for statistical tests:** XGBoost (champion model — selected based on lowest residual variability across wells).

---

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from scipy import stats as scipy_stats
from scipy.stats import shapiro, spearmanr
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sonic_ml_utils import plot_well_profile_and_hexabin, statistics

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

## 2. Directories and Configuration

In [ ]:
# Directories
RESULTS_DIR     = Path('../results')
COMPARISON_DIR  = RESULTS_DIR / 'comparison'
FIGURES_DIR     = COMPARISON_DIR / 'figures'
TABLES_DIR      = COMPARISON_DIR / 'tables'

for d in [FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURES      = ['GR', 'RT90', 'RHOB', 'NPHI']
REF_METHOD    = 'XGBoost'   # Champion model — reference for paired tests
N_WELLS       = 27          # Total wells in LOWO
ALPHA         = 0.05        # Significance level

print(f'Directories configured.')
print(f'Reference method : {REF_METHOD}')
print(f'Wells (LOWO)     : {N_WELLS}')
print(f'Alpha            : {ALPHA}')


## 3. Loading Results

In [ ]:
METRICS_FILES = {
    'XGBoost'     : RESULTS_DIR / 'xgboost'      / 'metrics' / 'lowo_xgboost_results.csv',
    'HistGB'      : RESULTS_DIR / 'histgb'       / 'metrics' / 'lowo_histgb_results.csv',
    'LightGBM'    : RESULTS_DIR / 'lightgbm'     / 'metrics' / 'lowo_lightgbm_results.csv',
    'RandomForest': RESULTS_DIR / 'randomforest' / 'metrics' / 'lowo_rf_results.csv',
    'MLP'         : RESULTS_DIR / 'mlp'          / 'metrics' / 'mlp_lowo_results.csv',
    'CNN'         : RESULTS_DIR / 'cnn'          / 'metrics' / 'cnn1d_lowo_results.csv',
}

results = {}
print('Loading metrics:')
for method, filepath in METRICS_FILES.items():
    if filepath.exists():
        results[method] = pd.read_csv(filepath)
        print(f'{method:20s} — {len(results[method])} wells')
    else:
        print(f'{method:20s} — not found: {filepath}')

print(f'\n{len(results)} methods loaded.')


## 4. Consolidating Results

In [ ]:
CATEGORIES = {
    'XGBoost'     : 'Machine Learning',
    'HistGB'      : 'Machine Learning',
    'LightGBM'    : 'Machine Learning',
    'RandomForest': 'Machine Learning',
    'MLP'         : 'Neural Network',
    'CNN'         : 'Neural Network',
}

all_results = []
for method, df in results.items():
    df_temp = df.copy()
    df_temp['Method']   = method
    df_temp['Category'] = CATEGORIES[method]
    df_temp = df_temp.rename(columns={'Well_Name': 'Well', 'R2': 'R²'})
    cols = ['Well', 'Method', 'Category', 'R²', 'RMSE', 'MAE']
    if 'Time_seconds' in df_temp.columns:
        cols.append('Time_seconds')
    all_results.append(df_temp[cols])

df_all = pd.concat(all_results, ignore_index=True)

print(f'Records    : {len(df_all):,}')
print(f'Methods    : {df_all["Method"].nunique()}')
print(f'Wells      : {df_all["Well"].nunique()}')
for method in [m for m in METRICS_FILES if m in df_all['Method'].values]:
    n = len(df_all[df_all['Method'] == method])
    print(f'  {method:20s}: {n} wells')


## 5. Overall Ranking

In [ ]:
def iqr(x):
    return x.quantile(0.75) - x.quantile(0.25)
iqr.__name__ = 'IQR'

agg_dict = {
    'R²'  : ['mean', 'median', 'std', 'min', 'max', iqr],
    'RMSE': ['mean', 'median', iqr],
    'MAE' : ['mean', 'median', iqr],
}
if 'Time_seconds' in df_all.columns:
    agg_dict['Time_seconds'] = ['mean', 'sum']

ranking = df_all.groupby(['Method', 'Category']).agg(agg_dict).round(4)
ranking.columns = ['_'.join(col).strip() for col in ranking.columns.values]
ranking = ranking.reset_index()
ranking = ranking.sort_values('R²_mean', ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1

print('FULL RANKING (sorted by mean R²):\n')
print(ranking[['Method', 'Category', 'R²_mean', 'R²_std', 'R²_IQR',
               'RMSE_mean', 'MAE_mean']].to_string())

# XGBoost tie-breaking justification
print('\n' + '='*70)
print('XGBoost SELECTION — TIE-BREAKING CRITERIA (gradient boosting group)')
print('='*70)
gb_methods = ['XGBoost', 'HistGB', 'LightGBM']
gb_rows = ranking[ranking['Method'].isin(gb_methods)][['Method','R²_mean','R²_std','R²_IQR','RMSE_mean']]
print(gb_rows.to_string(index=False))
print('\nJustification: XGBoost selected as champion based on lowest R² std')
print('and lowest R² IQR among the three statistically equivalent gradient')
print('boosting algorithms. Mean R² leadership alternates across executions')
print('(randomized hyperparameter search), confirming statistical equivalence.')


## 6. Bar Chart — Ranking

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

color_map   = {'Machine Learning': 'steelblue', 'Neural Network': 'coral'}
rank_sorted = ranking.sort_values('R²_mean', ascending=True)
colors      = [color_map[cat] for cat in rank_sorted['Category']]

ax.barh(rank_sorted['Method'], rank_sorted['R²_mean'],
        color=colors, edgecolor='black', linewidth=1.2)

for i, (_, row) in enumerate(rank_sorted.iterrows()):
    ax.text(row['R²_mean'] + 0.005, i, f'{row["R²_mean"]:.4f}',
            va='center', fontsize=10, fontweight='bold')

ax.axvline(x=0,   color='red',    linestyle='--', linewidth=1.5, alpha=0.6)
ax.axvline(x=0.5, color='orange', linestyle='--', linewidth=1.2, alpha=0.5)
ax.set_xlabel('Mean R² (LOWO)', fontsize=12, fontweight='bold')
ax.set_title('Method Ranking — DT (Sonic Log) Prediction\n'
             'Sergipe-Alagoas Basin | LOWO Validation | 27 wells',
             fontsize=13, fontweight='bold')
ax.set_xlim(left=0)
ax.grid(axis='x', alpha=0.3)
legend_elements = [mpatches.Patch(facecolor=c, edgecolor='black', label=l)
                   for l, c in color_map.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ranking_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Comparative Boxplot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

method_order = ranking.sort_values('R²_mean', ascending=False)['Method'].tolist()

sns.boxplot(data=df_all, x='Method', y='R²', order=method_order,
            palette='Set2', ax=ax, showmeans=True,
            meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

ax.axhline(y=0,    color='red',    linestyle='--', linewidth=2,   alpha=0.5, label='R²=0 (baseline)')
ax.axhline(y=0.5,  color='orange', linestyle='--', linewidth=1.5, alpha=0.5, label='R²=0.5')
ax.axhline(y=0.70, color='green',  linestyle=':',  linewidth=1.5, alpha=0.6, label='R²=0.70 (threshold)')

ax.set_xlabel('Method', fontsize=13, fontweight='bold')
ax.set_ylabel('R² (per well)', fontsize=13, fontweight='bold')
ax.set_title('R² Distribution by Method — LOWO (27 wells)\n'
             'Sergipe-Alagoas Basin',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'boxplot_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Identify Best and Worst Well

In [ ]:
df_ml_nn = df_all[df_all['Category'].isin(['Machine Learning', 'Neural Network'])].copy()
well_performance = df_ml_nn.groupby('Well')['R²'].mean().sort_values(ascending=False)

best_well  = well_performance.idxmax()
worst_well = well_performance.idxmin()

print(f'BEST WELL   : {best_well}  (mean R² ML+NN: {well_performance[best_well]:.4f})')
print(f'WORST WELL  : {worst_well}  (mean R² ML+NN: {well_performance[worst_well]:.4f})')

print('\nTOP 5 BEST:')
for i, (well, r2) in enumerate(well_performance.head(5).items(), 1):
    print(f'  {i}. {well:35s} R²={r2:.4f}')

print('\nTOP 5 WORST:')
for i, (well, r2) in enumerate(well_performance.tail(5).items(), 1):
    print(f'  {i}. {well:35s} R²={r2:.4f}')


## 9. Load Predictions

In [ ]:
PREDS_FILES = {
    'XGBoost'     : RESULTS_DIR / 'xgboost'      / 'predictions' / 'lowo_xgboost_predictions.csv',
    'HistGB'      : RESULTS_DIR / 'histgb'       / 'predictions' / 'lowo_histgb_predictions.csv',
    'LightGBM'    : RESULTS_DIR / 'lightgbm'     / 'predictions' / 'lowo_lightgbm_predictions.csv',
    'RandomForest': RESULTS_DIR / 'randomforest' / 'predictions' / 'lowo_rf_predictions.csv',
    'MLP'         : RESULTS_DIR / 'mlp'          / 'predictions' / 'lowo_mlp_predictions.csv',
    'CNN'         : RESULTS_DIR / 'cnn'          / 'predictions' / 'lowo_cnn_predictions.csv',
}

predictions = {}
print('Loading predictions:')
for method, filepath in PREDS_FILES.items():
    if filepath.exists():
        predictions[method] = pd.read_csv(filepath)
        df_p = predictions[method]
        print(f'{method:20s} — {len(df_p):,} samples, {df_p["Well_Name"].nunique()} wells')
    else:
        print(f'{method:20s} — not found: {filepath}')

print(f'\n{len(predictions)} methods with predictions loaded.')


## 10. Visualization — Best Well (all methods)

In [ ]:
df_original = pd.read_csv('../data/processed/wells_iqr_with_lithology.csv')

for method_name, df_pred in predictions.items():
    df_well = df_pred[df_pred['Well_Name'] == best_well].copy()
    if len(df_well) == 0:
        print(f'{method_name}: no data for {best_well}')
        continue

    if 'DEPTH' not in df_well.columns:
        depth_ref = (df_original[df_original['Well_Name'] == best_well]
                     [['Well_Name', 'DEPTH']].reset_index(drop=True))
        df_well = df_well.reset_index(drop=True)
        df_well['DEPTH'] = depth_ref['DEPTH']

    r2   = r2_score(df_well['DT_real'], df_well['DT_pred'])
    rmse = np.sqrt(mean_squared_error(df_well['DT_real'], df_well['DT_pred']))
    print(f'\n{method_name}  R²={r2:.4f}  RMSE={rmse:.2f}')

    well_safe = best_well.replace(' ', '_').replace('/', '-')
    save_path = str(FIGURES_DIR / f'best_well_{well_safe}_{method_name.lower()}.png')

    plot_well_profile_and_hexabin(
    predictions_df=df_well,
    well_name=None,
    figsize=(10, 8),
    cmap='YlGnBu',
    gridsize=70,
    save_path=save_path
    )


## 10b. Performance on the Best Well

In [ ]:
df_best_results = df_all[df_all['Well'] == best_well].sort_values('R²', ascending=False)

print(f'PERFORMANCE — {best_well}')
print(f'{"Rank":>5s} {"Method":>20s} {"Category":>20s} {"R²":>10s} {"RMSE":>10s} {"MAE":>10s}')
print('-' * 75)
for rank, (_, row) in enumerate(df_best_results.iterrows(), 1):
    print(f'{rank:>5d} {row["Method"]:>20s} {row["Category"]:>20s} '
          f'{row["R²"]:>10.4f} {row["RMSE"]:>10.2f} {row["MAE"]:>10.2f}')


## 11. Underperforming Wells & Threshold Justification (R² < 0.70)

### Threshold justification — three complementary pillars

**Pillar 1 — Natural discontinuity in the R² distribution**  
A visible gap exists between wells with R² ≥ 0.75 (majority) and those below. The R² = 0.70 cut is not arbitrary — it reflects a genuine discontinuity in the per-well results.

**Pillar 2 — Dual-criterion convergence**  
Two independent metrics (R² < 0.70 and RMSE > 6.0 µs/ft) identify the same set of problematic wells.

**Pillar 3 — DT variance analysis**  
Spearman correlation between per-well DT standard deviation and R² confirms that flagged wells are not cases of artificially penalized R² due to low target variance.

**Literature support:** Hair et al. (2019) — R² = 0.70 as the boundary between moderate and good predictive models in applied regression.

In [ ]:
# Pillar 1: Natural discontinuity
r2_ref = df_all[df_all['Method'] == REF_METHOD]['R²'].sort_values().values

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(r2_ref, bins=20, color='steelblue', edgecolor='black', alpha=0.75)
ax.axvline(0.70, color='red',    linestyle='--', linewidth=2,   label='R²=0.70 (threshold)')
ax.axvline(0.75, color='orange', linestyle=':',  linewidth=1.5, label='R²=0.75')
ax.set_xlabel('R² per well (LOWO)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of wells', fontsize=12, fontweight='bold')
ax.set_title(f'R² Distribution — {REF_METHOD} | 27 wells\n'
             'Natural discontinuity at R²=0.70', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'r2_distribution_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

# Distribution by range
bins_range   = [-np.inf, 0.50, 0.70, 0.75, np.inf]
labels_range = ['R² < 0.50', '0.50 ≤ R² < 0.70', '0.70 ≤ R² < 0.75', 'R² ≥ 0.75']
counts = pd.cut(pd.Series(r2_ref), bins=bins_range, labels=labels_range).value_counts().reindex(labels_range)
print('R² DISTRIBUTION BY RANGE (XGBoost — 27 wells):')
print(f'{"Range":>22s}  {"N wells":>8s}  {"% total":>8s}')
print('-' * 44)
for label, n in counts.items():
    print(f'{label:>22s}  {n:>8d}  {n/N_WELLS*100:>7.1f}%')


In [ ]:
# Pillar 2: Dual-criterion convergence
RMSE_THRESHOLD = 6.0  # µs/ft
R2_THRESHOLD   = 0.70

df_ref = df_all[df_all['Method'] == REF_METHOD].copy()
df_ref['flag_r2']   = df_ref['R²']   < R2_THRESHOLD
df_ref['flag_rmse'] = df_ref['RMSE'] > RMSE_THRESHOLD
df_ref['flag_dual'] = df_ref['flag_r2'] | df_ref['flag_rmse']

df_dual = df_ref[['Well', 'R²', 'RMSE', 'flag_r2', 'flag_rmse', 'flag_dual']].copy()
df_dual = df_dual.sort_values('R²')

print(f'DUAL CRITERION: R² < {R2_THRESHOLD} OR RMSE > {RMSE_THRESHOLD} µs/ft')
print(f'Reference method: {REF_METHOD}')
print()
print(f'{"Well":>35s}  {"R²":>8s}  {"RMSE":>8s}  {"R²<0.70":>8s}  {"RMSE>6.0":>9s}  {"Flagged":>8s}')
print('-' * 90)
for _, row in df_dual.iterrows():
    flag_r2   = '✅' if row['flag_r2']   else '  '
    flag_rmse = '✅' if row['flag_rmse'] else '  '
    flagged   = '⚠️' if row['flag_dual'] else '  '
    print(f'{row["Well"]:>35s}  {row["R²"]:>8.4f}  {row["RMSE"]:>8.3f}  '
          f'{flag_r2:>8s}  {flag_rmse:>9s}  {flagged:>8s}')

problematic = df_ref[df_ref['flag_dual']]['Well'].tolist()
print(f'\nProblematic wells (dual criterion): {len(problematic)}')
for w in problematic:
    print(f'  - {w}')

# Save table
df_dual.to_csv(TABLES_DIR / 'dual_criterion_threshold.csv', index=False)
print('\nTable saved: dual_criterion_threshold.csv')


In [ ]:
# Pillar 3: Spearman correlation std(DT) × R²
df_ref = df_all[df_all['Method'] == REF_METHOD].copy()

# Load predictions to compute per-well DT std
if REF_METHOD in predictions:
    df_pred_ref = predictions[REF_METHOD].copy()
    dt_std_per_well = (
        df_pred_ref.groupby('Well_Name')['DT_real']
        .std()
        .reset_index()
        .rename(columns={'Well_Name': 'Well', 'DT_real': 'DT_std'})
    )
    df_spearman = df_ref.merge(dt_std_per_well, on='Well', how='inner')

    rho, p_val = spearmanr(df_spearman['DT_std'], df_spearman['R²'])

    print('PILLAR 3 — Spearman Correlation: std(DT) × R²')
    print(f'  n wells  : {len(df_spearman)}')
    print(f'  Spearman ρ : {rho:.4f}')
    print(f'  p-value    : {p_val:.4f}')
    sig = 'significant' if p_val < ALPHA else 'NOT significant'
    print(f'  Interpretation: {sig} at α={ALPHA}')
    print()
    if p_val < ALPHA:
        print('  ✅ Confirmed: wells with higher DT variance tend to have higher R².')
        print('     The 4 flagged wells do NOT have unusually low DT variance —')
        print('     their low R² reflects genuine geological limitations.')
    else:
        print('  ⚠️  Correlation not significant — interpret threshold with caution.')

    # Scatter plot
    fig, ax = plt.subplots(figsize=(8, 6))
    colors_scatter = ['red' if w in problematic else 'steelblue'
                      for w in df_spearman['Well']]
    ax.scatter(df_spearman['DT_std'], df_spearman['R²'],
               c=colors_scatter, s=60, edgecolors='black', linewidths=0.7, zorder=3)
    ax.axhline(R2_THRESHOLD, color='red', linestyle='--', linewidth=1.5, alpha=0.7,
               label=f'R²={R2_THRESHOLD} (threshold)')
    ax.set_xlabel('DT Standard Deviation [µs/ft]', fontsize=12, fontweight='bold')
    ax.set_ylabel('R² LOWO', fontsize=12, fontweight='bold')
    ax.set_title(f'std(DT) vs R² — {REF_METHOD} | 27 wells\n'
                 f'Spearman ρ={rho:.3f}, p={p_val:.4f}', fontsize=12, fontweight='bold')
    legend_elements = [
        mpatches.Patch(facecolor='red',       edgecolor='black', label='Problematic (R²<0.70)'),
        mpatches.Patch(facecolor='steelblue', edgecolor='black', label='Good performance'),
    ]
    ax.legend(handles=legend_elements, fontsize=10)
    ax.grid(alpha=0.3)
    ax.text(0.02, 0.02, f'ρ={rho:.3f}\np={p_val:.4f}\nn={len(df_spearman)}',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'spearman_dt_std_vs_r2.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Save
    spearman_out = pd.DataFrame({'rho': [rho], 'p_value': [p_val],
                                  'n_wells': [len(df_spearman)], 'method': [REF_METHOD]})
    spearman_out.to_csv(TABLES_DIR / 'spearman_dt_std_r2.csv', index=False)
    print('\nTable saved: spearman_dt_std_r2.csv')
else:
    print(f'⚠️Predictions for {REF_METHOD} not loaded — run Section 9 first.')


In [ ]:
print(df_spearman[['Well', 'DT_std', 'R²']].sort_values('R²').head(8).to_string(index=False))

In [ ]:
# Consistency check: all 4 tree-based algorithms flag the same wells
tree_methods = ['XGBoost', 'HistGB', 'LightGBM', 'RandomForest']
available_tree = [m for m in tree_methods if m in df_all['Method'].values]

print('CROSS-ALGORITHM CONSISTENCY CHECK')
print(f'Threshold: R² < {R2_THRESHOLD}')
print()

flagged_per_method = {}
for method in available_tree:
    df_m = df_all[df_all['Method'] == method]
    flagged = set(df_m[df_m['R²'] < R2_THRESHOLD]['Well'].tolist())
    flagged_per_method[method] = flagged
    print(f'  {method:20s}: {len(flagged)} wells flagged — {sorted(flagged)}')

# Wells flagged by ALL tree-based methods
flagged_all = set.intersection(*flagged_per_method.values())
print(f'\nFlagged by ALL tree-based methods ({len(flagged_all)} wells):')
for w in sorted(flagged_all):
    print(f'  - {w}')

consistency = all(s == flagged_per_method[available_tree[0]] for s in flagged_per_method.values())
print(f'\nPerfect consistency across algorithms: {"YES" if consistency else "NO"}')
if consistency:
    print('Strong evidence that failures are geological, not methodological.')



## 12. Worst Well Analysis

In [ ]:
df_worst_results = df_all[df_all['Well'] == worst_well].sort_values('R²', ascending=False)

print(f'PERFORMANCE — {worst_well}')
print(f'{"Rank":>5s} {"Method":>20s} {"Category":>20s} {"R²":>10s} {"RMSE":>10s}')
print('-' * 65)
for rank, (_, row) in enumerate(df_worst_results.iterrows(), 1):
    print(f'{rank:>5d} {row["Method"]:>20s} {row["Category"]:>20s} '
          f'{row["R²"]:>10.4f} {row["RMSE"]:>10.2f}')


## 13. R² Distribution by Method (ML and NN)

In [ ]:
methods_to_plot = ['XGBoost', 'HistGB', 'LightGBM', 'RandomForest', 'MLP', 'CNN']
available_plot  = [m for m in methods_to_plot if m in df_all['Method'].values]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, method in enumerate(available_plot):
    ax = axes[idx]
    df_m = df_all[df_all['Method'] == method]

    r2_mean   = df_m['R²'].mean()
    rmse_mean = df_m['RMSE'].mean()
    mae_mean  = df_m['MAE'].mean()

    ax.hist(df_m['R²'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(r2_mean,    color='red',   linestyle='--', linewidth=2, label=f'Mean: {r2_mean:.4f}')
    ax.axvline(R2_THRESHOLD, color='black', linestyle=':',  linewidth=1.5, alpha=0.7,
               label=f'Threshold: {R2_THRESHOLD}')
    ax.set_xlabel('R²', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency (wells)', fontsize=11, fontweight='bold')
    ax.set_title(method, fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.text(0.98, 0.98, f'RMSE: {rmse_mean:.2f}\nMAE: {mae_mean:.2f}',
            transform=ax.transAxes, fontsize=9, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Hide unused subplots
for idx in range(len(available_plot), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('R² distribution by method (ML and Neural Networks)',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'r2_distribution_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()


## 14. Final Summary Table

In [ ]:
table_data = []
for _, row in ranking.iterrows():
    method = row['Method']
    df_m   = df_all[df_all['Method'] == method]
    n_prob = (df_m['R²'] < R2_THRESHOLD).sum()
    row_data = {
        'Method'            : method,
        'Category'          : row['Category'],
        'R² (mean±std)'     : f"{row['R²_mean']:.4f} ± {row['R²_std']:.4f}",
        'R² IQR'            : f"{row['R²_IQR']:.4f}",
        'RMSE (mean)'       : f"{row['RMSE_mean']:.2f}",
        'RMSE IQR'          : f"{row['RMSE_IQR']:.2f}",
        'MAE (mean)'        : f"{row['MAE_mean']:.2f}",
        'MAE IQR'           : f"{row['MAE_IQR']:.2f}",
        'Wells R²<0.70 (%)' : f"{n_prob/len(df_m)*100:.1f}",
    }
    if 'Time_seconds' in df_all.columns and 'Time_seconds_sum' in row.index:
        row_data['LOWO Time (min)'] = f"{row['Time_seconds_sum']/60:.1f}"
        row_data['Time/well (s)']   = f"{row['Time_seconds_mean']:.1f}"
    table_data.append(row_data)

df_table = pd.DataFrame(table_data)
print(df_table.to_string(index=False))

latex_code = df_table.to_latex(
    index=False, column_format='llcccccccc',
    caption='Comparison of all methods for DT prediction using LOWO validation — '
            'Sergipe-Alagoas Basin (27 wells).',
    label='tab:all_methods_comparison'
)
with open(TABLES_DIR / 'table_all_methods.tex', 'w') as f:
    f.write(latex_code)
print('\nLaTeX table saved: table_all_methods.tex')


In [ ]:
# Execution time ranking
if 'Time_seconds' in df_all.columns:
    time_summary = (
        df_all[df_all['Time_seconds'].notna()]
        .groupby('Method')['Time_seconds']
        .agg(total_s='sum', mean_s='mean', n='count')
        .assign(total_min=lambda x: x['total_s'] / 60)
        .sort_values('total_min')
        .reset_index()
    )
    print('LOWO EXECUTION TIME RANKING')
    print('=' * 65)
    print(f'{"Rank":>5s}  {"Method":>20s}  {"Total time":>12s}  {"Time/well":>12s}')
    print('-' * 65)
    for rank, (_, row) in enumerate(time_summary.iterrows(), 1):
        print(f'{rank:>5d}  {row["Method"]:>20s}  '
              f'{row["total_min"]:>10.1f} min  '
              f'{row["mean_s"]:>10.1f} s/well')
else:
    print('Time_seconds not available in CSVs.')


## 15. Executive Summary

In [ ]:
top3    = ranking.head(3)
ml_best = ranking[ranking['Category'] == 'Machine Learning'].iloc[0]
nn_best = ranking[ranking['Category'] == 'Neural Network'].iloc[0]
delta_ml_nn = ml_best['R²_mean'] - nn_best['R²_mean']

mlm  = ml_best['Method']
mlr2 = ml_best['R²_mean']
mls  = ml_best['R²_std']
mliq = ml_best['R²_IQR']
nnm  = nn_best['Method']
nnr2 = nn_best['R²_mean']
mlpr2 = ranking[ranking['Method']=='MLP']['R²_mean'].values[0]
mlpiq = ranking[ranking['Method']=='MLP']['R²_IQR'].values[0]
cnnr2 = ranking[ranking['Method']=='CNN']['R²_mean'].values[0]
cnniq = ranking[ranking['Method']=='CNN']['R²_IQR'].values[0]

# XGBoost tie-breaking: std and IQR comparison
xgb_row = ranking[ranking['Method'] == 'XGBoost'].iloc[0]
xgb_std = xgb_row['R²_std']
xgb_iqr = xgb_row['R²_IQR']

print('EXECUTIVE SUMMARY')
print('=' * 80)
print('\nTOP 3 METHODS:')
for idx, row in top3.iterrows():
    print(f'  {idx}º  {row["Method"]:20s}  ({row["Category"]})')
    print(f'      R²={row["R²_mean"]:.4f} ± {row["R²_std"]:.4f}  '
          f'IQR={row["R²_IQR"]:.4f}  RMSE={row["RMSE_mean"]:.2f}  MAE={row["MAE_mean"]:.2f}')

print('\nPERFORMANCE BY CATEGORY:')
cat_perf = df_all.groupby('Category')['R²'].agg(['mean', 'std', 'min', 'max'])
print(cat_perf.to_string())

print(f"""
MAIN FINDINGS:

1. CHAMPION MODEL: XGBoost
   R²={mlr2:.4f} ± {mls:.4f}  |  IQR={mliq:.4f}
   Selected over HistGB and LightGBM by lowest residual variability
   (std={xgb_std:.4f}, IQR={xgb_iqr:.4f}). Mean R² leadership alternates
   across RandomizedSearchCV executions — statistical equivalence confirmed.

2. ML vs NEURAL NETWORKS:
   Best ML ({mlm}): R²={mlr2:.4f}
   Best NN ({nnm}): R²={nnr2:.4f}
   Delta R²: {delta_ml_nn:+.4f}

3. NEURAL NETWORKS:
   MLP: R²={mlpr2:.4f}  IQR={mlpiq:.4f}
   CNN: R²={cnnr2:.4f}  IQR={cnniq:.4f}
""")

## 16. Confidence Intervals (95%)

In [ ]:
# Normality check on per-well R² (justifies use of t-test)
print('SHAPIRO-WILK TEST — per-well R² (justification for parametric t-test)')
print(f'{"Method":>20s}  {"stat":>8s}  {"p-value":>10s}  {"Distribution":>15s}')
print('-' * 60)
for method in [m for m in ['XGBoost','HistGB','LightGBM','RandomForest','MLP','CNN']
               if m in df_all['Method'].values]:
    r2_vals = df_all[df_all['Method'] == method]['R²'].values
    stat, p = shapiro(r2_vals)
    dist = 'Normal' if p > ALPHA else 'Non-normal'
    print(f'{method:>20s}  {stat:>8.4f}  {p:>10.4f}  {dist:>15s}')
print()
print('Note: Wilcoxon signed-rank test used as non-parametric complement')
print('regardless of normality. Both tests reported; consensus required.')


In [ ]:
confidence_results = []
for method in ranking['Method']:
    r2_scores = df_all[df_all['Method'] == method]['R²'].values
    ci = statistics.calculate_confidence_interval(r2_scores, confidence=0.95)
    confidence_results.append({
        'Method'  : method,
        'Mean'    : ci['mean'],
        'Lower_CI': ci['lower_bound'],
        'Upper_CI': ci['upper_bound'],
        'Width'   : ci['upper_bound'] - ci['lower_bound']
    })

df_ci = pd.DataFrame(confidence_results).sort_values('Mean', ascending=False)

print(f'{"Method":>20s} {"Mean R²":>10s} {"95% CI":>30s} {"Width":>10s}')
print('-' * 74)
for _, row in df_ci.iterrows():
    ci_str = f'[{row["Lower_CI"]:.4f}, {row["Upper_CI"]:.4f}]'
    print(f'{row["Method"]:>20s} {row["Mean"]:>10.4f} {ci_str:>30s} {row["Width"]:>10.4f}')

df_ci.to_csv(TABLES_DIR / 'confidence_intervals_95.csv', index=False)
print('\nTable saved: confidence_intervals_95.csv')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

method_order = ['XGBoost', 'LightGBM', 'HistGB', 'RandomForest', 'MLP', 'CNN']
available_order = [m for m in method_order if m in df_all['Method'].values]

palette = {m: ('steelblue' if df_all[df_all['Method'] == m]['Category'].iloc[0] == 'Machine Learning'
               else 'coral')
           for m in available_order}

sns.violinplot(data=df_all, x='Method', y='R²', order=available_order,
               palette=palette, inner=None, alpha=0.4, ax=ax)

sns.swarmplot(data=df_all, x='Method', y='R²', order=available_order,
              palette=palette, size=7, edgecolor='black', linewidth=0.7, ax=ax)

ax.axhline(y=0.70, color='red',    linestyle='--', linewidth=1.5, alpha=0.7,
           label='R²=0.70 threshold')
ax.axhline(y=0.50, color='orange', linestyle='--', linewidth=1.2, alpha=0.5,
           label='R²=0.50')

ax.set_xlabel('Method',      fontsize=13, fontweight='bold')
ax.set_ylabel('R² per well', fontsize=13, fontweight='bold')
ax.set_title('R² Distribution by Method — LOWO (27 wells)\n'
             'Sergipe-Alagoas Basin | ML and Neural Network Methods',
             fontsize=14, fontweight='bold')

label_map = {'Machine Learning': 'Machine Learning', 'Neural Network': 'Neural Network'}
legend_elements = [
    mpatches.Patch(facecolor=c, edgecolor='black', label=label_map[l])
    for l, c in color_map.items()
] + [
    plt.Line2D([0], [0], color='red',    linestyle='--', label='R²=0.70 threshold'),
    plt.Line2D([0], [0], color='orange', linestyle='--', label='R²=0.50'),
    plt.scatter([], [], s=50, c='gray', edgecolors='black', linewidth=0.7,
                label='Well'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'violin_r2_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()

## 17. Paired Statistical Tests

Comparisons performed with XGBoost as reference (champion model):
- XGBoost vs HistGB
- XGBoost vs LightGBM
- XGBoost vs RandomForest
- XGBoost vs MLP *(best tree-based vs best neural network)*
- MLP vs CNN *(within neural network group)*

Both paired t-test and Wilcoxon signed-rank test applied (α=0.05, n=27 wells per pair). Consensus requires both tests to agree.

In [ ]:
comparisons = [
    ('XGBoost', 'HistGB',        'XGBoost vs HistGB'),
    ('XGBoost', 'LightGBM',      'XGBoost vs LightGBM'),
    ('XGBoost', 'RandomForest',  'XGBoost vs RandomForest'),
    ('XGBoost', 'MLP',           'Best ML vs Best NN'),
    ('MLP',     'CNN',           'MLP vs CNN'),
]

test_results = []
print('PAIRED STATISTICAL TESTS (t-test + Wilcoxon, α=0.05)')
print('Reference: XGBoost (champion model)')
print('=' * 80)

for m1, m2, desc in comparisons:
    if m1 not in df_all['Method'].values or m2 not in df_all['Method'].values:
        print(f'\n{desc}: {m1} vs {m2} — ⚠️ method not found, skipping.')
        continue

    df_m1 = df_all[df_all['Method'] == m1].sort_values('Well')
    df_m2 = df_all[df_all['Method'] == m2].sort_values('Well')

    # Keep only wells present in both
    common_wells = set(df_m1['Well']) & set(df_m2['Well'])
    df_m1 = df_m1[df_m1['Well'].isin(common_wells)].sort_values('Well')
    df_m2 = df_m2[df_m2['Well'].isin(common_wells)].sort_values('Well')

    scores1 = df_m1['R²'].values
    scores2 = df_m2['R²'].values
    wells1  = df_m1['Well'].values
    wells2  = df_m2['Well'].values

    test = statistics.statistical_test_models(
        scores1, scores2, alpha=ALPHA,
        wells1=wells1, wells2=wells2
    )

    # Cohen's d (effect size)
    diffs  = scores1 - scores2
    cohens_d = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else 0.0

    test_results.append({
        'Comparison'        : f'{m1} vs {m2}',
        'Description'       : desc,
        'Δ R²'              : test['mean_difference'],
        "Cohen's d"         : cohens_d,
        't-stat'            : test['t_statistic'],
        'p-value (t-test)'  : test['p_value_ttest'],
        'p-value (Wilcoxon)': test['p_value_wilcoxon'],
        'N wells'           : test['n_wells'],
        'Significant'       : test['significant'],
    })

    sig  = 'SIGNIFICANT' if test['significant'] else 'NOT SIGNIFICANT'
    d_interp = ('negligible' if abs(cohens_d) < 0.2 else
                'small'      if abs(cohens_d) < 0.5 else
                'medium'     if abs(cohens_d) < 0.8 else 'large')
    print(f'\n{desc}: {m1} vs {m2}  (n={test["n_wells"]} wells)')
    print(f'  Mean Δ R²  : {test["mean_difference"]:+.4f}')
    print(f"  Cohen's d  : {cohens_d:+.4f} ({d_interp} effect)")
    print(f'  t-test     : p={test["p_value_ttest"]:.4f} → {"sig." if test["significant_ttest"] else "n.s."}')
    print(f'  Wilcoxon   : p={test["p_value_wilcoxon"]:.4f} → {"sig." if test["significant_wilcoxon"] else "n.s."}')
    print(f'  Consensus  : {sig}')

print('\n' + '='*80)
df_tests = pd.DataFrame(test_results)
print('\nSUMMARY TABLE:')
print(df_tests.to_string(index=False))

df_tests.to_csv(TABLES_DIR / 'paired_statistical_tests.csv', index=False)
print('\nTable saved: paired_statistical_tests.csv')


## 18. Residual Analysis — XGBoost (Champion Model)

In [ ]:
df_xgb_m = df_all[df_all['Method'] == REF_METHOD].copy()
rmse_values = df_xgb_m['RMSE'].values
mae_values  = df_xgb_m['MAE'].values

print(f'RESIDUAL ANALYSIS — {REF_METHOD}')
print(f'  Mean R²   : {df_xgb_m["R²"].mean():.4f}')
print(f'  Mean RMSE : {rmse_values.mean():.2f} µs/ft')
print(f'  Mean MAE  : {mae_values.mean():.2f} µs/ft')

shapiro_stat_rmse, shapiro_p_rmse = scipy_stats.shapiro(rmse_values)
shapiro_stat_r2,   shapiro_p_r2   = scipy_stats.shapiro(df_xgb_m['R²'].values)
print(f'\nShapiro-Wilk (RMSE) : stat={shapiro_stat_rmse:.4f}  p={shapiro_p_rmse:.4f}  '
      f'({"Normal" if shapiro_p_rmse > ALPHA else "Non-normal"})')
print(f'Shapiro-Wilk (R²)   : stat={shapiro_stat_r2:.4f}  p={shapiro_p_r2:.4f}  '
      f'({"Normal" if shapiro_p_r2 > ALPHA else "Non-normal"})')

Q1, Q3 = np.percentile(rmse_values, [25, 75])
n_outliers = (rmse_values > Q3 + 1.5*(Q3-Q1)).sum()
cv = rmse_values.std() / rmse_values.mean() * 100
print(f'CV RMSE : {cv:.1f}%  |  RMSE outliers: {n_outliers}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
ax1.hist(rmse_values, bins=15, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(rmse_values.mean(),    color='red',    linestyle='--', lw=2,
            label=f'Mean: {rmse_values.mean():.2f}')
ax1.axvline(np.median(rmse_values), color='orange', linestyle='--', lw=2,
            label=f'Median: {np.median(rmse_values):.2f}')
ax1.set_xlabel('RMSE (µs/ft)'); ax1.set_ylabel('Frequency')
ax1.set_title(f'RMSE Distribution — {REF_METHOD}')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

ax2 = axes[0, 1]
ax2.hist(df_xgb_m['R²'].values, bins=15, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(df_xgb_m['R²'].mean(), color='red', linestyle='--', lw=2,
            label=f'Mean: {df_xgb_m["R²"].mean():.4f}')
ax2.axvline(R2_THRESHOLD, color='black', linestyle=':', lw=1.5, alpha=0.8,
            label=f'Threshold: {R2_THRESHOLD}')
ax2.set_xlabel('R²'); ax2.set_ylabel('Frequency')
ax2.set_title(f'R² Distribution — {REF_METHOD}')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

ax3 = axes[1, 0]
scipy_stats.probplot(rmse_values, dist='norm', plot=ax3)
ax3.set_title('Q-Q Plot — RMSE'); ax3.grid(alpha=0.3)

ax4 = axes[1, 1]
bp = ax4.boxplot([rmse_values, mae_values], labels=['RMSE', 'MAE'],
                  patch_artist=True, showmeans=True,
                  meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('coral')
ax4.set_ylabel('Error (µs/ft)')
ax4.set_title(f'Error Boxplot — {REF_METHOD}')
ax4.grid(axis='y', alpha=0.3)

plt.suptitle(f'Residual Analysis — {REF_METHOD} | 27 wells',
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'residual_analysis_{REF_METHOD}.png', dpi=150, bbox_inches='tight')
plt.show()


## 19. Heatmap — R² per Well × Algorithm

In [ ]:
all_methods_hm = ['XGBoost', 'HistGB', 'LightGBM', 'RandomForest', 'MLP', 'CNN']
available_hm   = [m for m in all_methods_hm if m in df_all['Method'].values]
n_tree_hm      = sum(1 for m in available_hm if CATEGORIES[m] == 'Machine Learning')
n_total_hm     = len(available_hm)

df_ml_hm = df_all[df_all['Method'].isin(available_hm)].copy()
df_pivot  = df_ml_hm.pivot(index='Well', columns='Method', values='R²')
df_pivot  = df_pivot[available_hm]
order_hm  = df_pivot[REF_METHOD].sort_values(ascending=False).index
df_pivot  = df_pivot.loc[order_hm]
n_wells_hm = len(df_pivot)

cmap_hm = mcolors.LinearSegmentedColormap.from_list(
    'r2_custom',
    [(0.00, '#E74C3C'), (0.50, '#E67E22'), (0.70, '#F1C40F'),
     (0.75, '#27AE60'), (1.00, '#1A5C35')]
)
norm_hm = mcolors.Normalize(vmin=0.0, vmax=1.0)

fig, ax = plt.subplots(figsize=(10, max(8, n_wells_hm * 0.45)))

for col_idx, method in enumerate(available_hm):
    for row_idx, well in enumerate(df_pivot.index):
        val = df_pivot.loc[well, method]
        color = cmap_hm(norm_hm(val)) if not np.isnan(val) else (0.9, 0.9, 0.9, 1.0)
        rect  = plt.Rectangle([col_idx - 0.5, row_idx - 0.5], 1, 1,
                               facecolor=color, edgecolor='white', linewidth=0.5)
        ax.add_patch(rect)
        if not np.isnan(val):
            txt_color = 'white' if val < 0.55 else 'black'
            ax.text(col_idx, row_idx, f'{val:.3f}',
                    ha='center', va='center', fontsize=7.5,
                    fontweight='bold', color=txt_color)

# Separator: good / problematic wells
n_good_hm = n_wells_hm - len(problematic)
ax.axhline(n_good_hm - 0.5, color='black', linewidth=2.0, linestyle='-', zorder=6)

ax.set_xticks(range(n_total_hm))
ax.set_xticklabels(available_hm, fontsize=11, fontweight='bold')
ax.set_yticks(range(n_wells_hm))
ax.set_yticklabels(df_pivot.index, fontsize=8.5)
ax.tick_params(length=0)
ax.set_xlim(-0.5, n_total_hm - 0.5)
ax.set_ylim(-0.5, n_wells_hm - 0.5)

# Group annotations
x_tree = (n_tree_hm / 2 - 0.5) / (n_total_hm - 1)
x_nn   = (n_tree_hm + (n_total_hm - n_tree_hm) / 2 - 0.5) / (n_total_hm - 1)
ax.annotate('Tree-Based', xy=(x_tree, 1.022), xycoords='axes fraction',
            ha='center', fontsize=10, fontweight='bold', color='#1A5C35')
ax.annotate('Neural Networks', xy=(x_nn, 1.022), xycoords='axes fraction',
            ha='center', fontsize=10, fontweight='bold', color='#2980B9')

sm   = plt.cm.ScalarMappable(cmap=cmap_hm, norm=norm_hm)
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('R² LOWO', fontsize=11)
# cbar.ax.axhline(0.70, color='black', linewidth=1.5, linestyle='--')
# cbar.ax.axhline(0.75, color='black', linewidth=1.0, linestyle=':')

legend_elements = [
    mpatches.Patch(facecolor='#E74C3C', label='R² < 0.50'),
    mpatches.Patch(facecolor='#E67E22', label='0.50 ≤ R² < 0.70'),
    mpatches.Patch(facecolor='#F1C40F', label='0.70 ≤ R² < 0.75'),
    mpatches.Patch(facecolor='#27AE60', label='R² ≥ 0.75'),
]
ax.legend(handles=legend_elements, loc='lower right',
          bbox_to_anchor=(1.0, -0.10), ncol=2, fontsize=8.5, framealpha=0.9)

ax.set_title('R² LOWO per Well — ML Methods and Neural Networks\n'
             'Sergipe-Alagoas Basin | 27 wells',
             fontsize=13, pad=22)
ax.set_xlabel('Algorithm', fontsize=11, labelpad=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'heatmap_r2_well_algorithm.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nProblematic wells (R² < {R2_THRESHOLD} on {REF_METHOD}): {problematic}')
print(f'Cross-algorithm consistency: all tree-based methods fail on the same {len(problematic)} wells')


In [ ]:
# Heatmap transposed: Wells (x) × Algorithms (y) — PowerPoint version
# Same R² ordering as Section 19 (wells sorted by XGBoost R², descending)

fig, ax = plt.subplots(figsize=(max(14, n_wells_hm * 0.55),8))

for row_idx, method in enumerate(available_hm):
    for col_idx, well in enumerate(df_pivot.index):
        val = df_pivot.loc[well, method]
        color = cmap_hm(norm_hm(val)) if not np.isnan(val) else (0.9, 0.9, 0.9, 1.0)
        rect  = plt.Rectangle([col_idx - 0.5, row_idx - 0.5], 1, 1,
                               facecolor=color, edgecolor='white', linewidth=0.5)
        ax.add_patch(rect)
        if not np.isnan(val):
            txt_color = 'white' if val < 0.55 else 'black'
            ax.text(col_idx, row_idx, f'{val:.2f}',
                    ha='center', va='center', fontsize=7.0,
                    fontweight='bold', color=txt_color)

# Separator: tree-based / neural networks (horizontal line)
ax.axhline(n_tree_hm - 0.5, color='black', linewidth=2.0, linestyle='-', zorder=6)

# Axes
ax.set_yticks(range(len(available_hm)))
ax.set_yticklabels(available_hm, fontsize=11, fontweight='bold')
ax.set_xticks(range(n_wells_hm))
# well_labels = [w.replace('3-BRSA-', '').replace('1-BRSA-', '') for w in df_pivot.index]
# ax.set_xticklabels(well_labels, rotation=45, ha='right', fontsize=8)
ax.set_xticklabels(df_pivot.index.tolist(), rotation=45, ha='right', fontsize=8)
ax.tick_params(length=0)
ax.set_xlim(-0.5, n_wells_hm - 0.5)
ax.set_ylim(-0.5, len(available_hm) - 0.5)

# Group annotations on y-axis
y_tree = (n_tree_hm / 2 - 0.5) / (len(available_hm) - 1)
y_nn   = (n_tree_hm + (len(available_hm) - n_tree_hm) / 2 - 0.5) / (len(available_hm) - 1)
# ax.annotate('Tree-Based', xy=(-0.01, y_tree), xycoords='axes fraction',
#             ha='right', va='center', fontsize=9, fontweight='bold',
#             color='#1A5C35', rotation=90)
# ax.annotate('Neural\nNetworks', xy=(-0.01, y_nn), xycoords='axes fraction',
#             ha='right', va='center', fontsize=9, fontweight='bold',
#             color='#2980B9', rotation=90)

# Colorbar
sm_t   = plt.cm.ScalarMappable(cmap=cmap_hm, norm=norm_hm)
cbar_t = fig.colorbar(sm_t, ax=ax, fraction=0.02, pad=0.01)
cbar_t.set_label('R² LOWO', fontsize=10)
# cbar_t.ax.axhline(0.70, color='black', linewidth=1.5, linestyle='--')
# cbar_t.ax.axhline(0.75, color='black', linewidth=1.0, linestyle=':')

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#E74C3C', label='R² < 0.50'),
    mpatches.Patch(facecolor='#E67E22', label='0.50 ≤ R² < 0.70'),
    mpatches.Patch(facecolor='#F1C40F', label='0.70 ≤ R² < 0.75'),
    mpatches.Patch(facecolor='#27AE60', label='R² ≥ 0.75'),
]
ax.legend(handles=legend_elements, loc='upper right',
          bbox_to_anchor=(1.0, -0.14), ncol=4, fontsize=8.5, framealpha=0.9)

ax.set_title('R² LOWO per Well — ML Methods and Neural Networks\n'
             'Sergipe-Alagoas Basin | 27 wells | Sorted by XGBoost R² (descending)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Well', fontsize=11, fontweight='bold', labelpad=8)
ax.set_ylabel('Algorithm', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'heatmap_r2_well_algorithm_pptx.png',
            dpi=300, bbox_inches='tight')
plt.show()

## 20. Supplementary Material for Article

In [ ]:
# Scatter hexbin
df_xgb_pred = predictions[REF_METHOD].copy()

r2_global   = r2_score(df_xgb_pred['DT_real'], df_xgb_pred['DT_pred'])
rmse_global = np.sqrt(mean_squared_error(df_xgb_pred['DT_real'], df_xgb_pred['DT_pred']))
mae_global  = mean_absolute_error(df_xgb_pred['DT_real'], df_xgb_pred['DT_pred'])
bias_global = (df_xgb_pred['DT_pred'] - df_xgb_pred['DT_real']).mean()
n_total_pred = len(df_xgb_pred)

lims = [df_xgb_pred['DT_real'].min() - 2, df_xgb_pred['DT_real'].max() + 2]

fig, ax = plt.subplots(figsize=(8, 8))
hb = ax.hexbin(df_xgb_pred['DT_real'], df_xgb_pred['DT_pred'],
               gridsize=70, cmap='YlOrRd', mincnt=1, linewidths=0.2)
cbar = plt.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Sample Count', fontsize=11)
ax.plot(lims, lims, 'b-', lw=1.8, label='1:1 reference', zorder=5)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Measured DT [µs/ft]', fontsize=13, fontweight='bold')
ax.set_ylabel('Predicted DT [µs/ft]', fontsize=13, fontweight='bold')
ax.set_title(f'{REF_METHOD} — Measured vs Predicted DT\n'
             'Sergipe-Alagoas Basin | LOWO Validation | 27 wells',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.2)
ax.set_aspect('equal')
ax.text(0.97, 0.05,
        f'R² = {r2_global:.4f}\nRMSE = {rmse_global:.2f} µs/ft\n'
        f'MAE = {mae_global:.2f} µs/ft\nBias = {bias_global:+.2f} µs/ft\nN = {n_total_pred:,}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=10.5,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray', linewidth=0.8))
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'scatter_hexbin_{REF_METHOD.lower()}_publication.png',
            dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure saved at dpi=300 (publication resolution)')


In [ ]:
# Error distribution
df_xgb_pred = predictions[REF_METHOD].copy()
df_xgb_pred['residual']  = df_xgb_pred['DT_pred'] - df_xgb_pred['DT_real']
df_xgb_pred['abs_error'] = df_xgb_pred['residual'].abs()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.hist(df_xgb_pred['residual'], bins=100, color='steelblue',
         edgecolor='none', alpha=0.85, density=True)
ax1.axvline(0, color='red', lw=1.5, linestyle='--', label='Zero bias')
ax1.axvline(df_xgb_pred['residual'].mean(), color='orange', lw=1.5,
            linestyle='--', label=f'Mean = {df_xgb_pred["residual"].mean():+.2f} µs/ft')
ax1.set_xlabel('Residual (Predicted − Measured) [µs/ft]', fontsize=12, fontweight='bold')
ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
ax1.set_title('Residual Distribution', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
p5, p95 = df_xgb_pred['residual'].quantile([0.05, 0.95])
ax1.text(0.97, 0.97,
         f'P5  = {p5:+.2f} µs/ft\nP95 = {p95:+.2f} µs/ft\n'
         f'Std = {df_xgb_pred["residual"].std():.2f} µs/ft',
         transform=ax1.transAxes, ha='right', va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax2.hist(df_xgb_pred['abs_error'], bins=100, color='steelblue',
         edgecolor='none', alpha=0.85, density=True)
ax2.axvline(df_xgb_pred['abs_error'].median(), color='red', lw=1.5,
            linestyle='--', label=f'Median = {df_xgb_pred["abs_error"].median():.2f} µs/ft')
ax2.axvline(df_xgb_pred['abs_error'].mean(), color='orange', lw=1.5,
            linestyle='--', label=f'Mean = {df_xgb_pred["abs_error"].mean():.2f} µs/ft')
ax2.set_xlabel('Absolute Error [µs/ft]', fontsize=12, fontweight='bold')
ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
ax2.set_title('Absolute Error Distribution', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
p50, p75, p90 = df_xgb_pred['abs_error'].quantile([0.50, 0.75, 0.90])
ax2.text(0.97, 0.97,
         f'P50 = {p50:.2f} µs/ft\nP75 = {p75:.2f} µs/ft\nP90 = {p90:.2f} µs/ft',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

fig.suptitle(f'{REF_METHOD} — Error Distribution | LOWO Validation\n'
             'Sergipe-Alagoas Basin | 27 wells',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'error_distribution_{REF_METHOD.lower()}_publication.png',
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Log profile comparison: best well vs problematic well
# Update these well names after running Section 8 if they differ
profile_best  = best_well
profile_worst = worst_well

fig, axes = plt.subplots(1, 4, figsize=(18, 10), sharey=False)

for col, (well, label) in enumerate([
    (profile_best,  'Best Well'),
    (profile_worst, 'Problematic Well'),
]):
    df_w   = predictions[REF_METHOD][predictions[REF_METHOD]['Well_Name'] == well].copy()
    r2_w   = r2_score(df_w['DT_real'], df_w['DT_pred'])
    rmse_w = np.sqrt(mean_squared_error(df_w['DT_real'], df_w['DT_pred']))
    bias_w = (df_w['DT_pred'] - df_w['DT_real']).mean()

    if 'DEPTH' in df_w.columns:
        df_w = df_w.sort_values('DEPTH')

    ax_prof = axes[col * 2]
    ax_hex  = axes[col * 2 + 1]

    if 'DEPTH' in df_w.columns:
        ax_prof.plot(df_w['DT_real'], df_w['DEPTH'], color='black',   lw=0.8, label='Measured DT')
        ax_prof.plot(df_w['DT_pred'], df_w['DEPTH'], color='crimson', lw=0.8, label='Predicted DT', alpha=0.85)
        ax_prof.invert_yaxis()
        ax_prof.set_ylabel('Depth [m]', fontsize=10, fontweight='bold')
    ax_prof.set_xlabel('DT [µs/ft]', fontsize=10, fontweight='bold')
    ax_prof.set_title(f'{label}\n{well}\nR²={r2_w:.3f}', fontsize=10, fontweight='bold')
    ax_prof.legend(fontsize=8, loc='lower right')
    ax_prof.grid(alpha=0.3)

    lims_w = [min(df_w['DT_real'].min(), df_w['DT_pred'].min()) - 2,
              max(df_w['DT_real'].max(), df_w['DT_pred'].max()) + 2]

    hb = ax_hex.hexbin(df_w['DT_real'], df_w['DT_pred'],
                       gridsize=100, cmap='YlOrRd', mincnt=1,
                       extent=lims_w + lims_w)
    plt.colorbar(hb, ax=ax_hex, label='Sample count')
    ax_hex.plot(lims_w, lims_w, 'k-', lw=1.5, label='1:1 (ideal)')
    ax_hex.set_xlim(lims_w)
    ax_hex.set_ylim(lims_w)
    ax_hex.set_xlabel('Measured DT [µs/ft]',   fontsize=10, fontweight='bold')
    ax_hex.set_ylabel('Predicted DT [µs/ft]',  fontsize=10, fontweight='bold')
    ax_hex.set_title(f'Hexbin — {label}',       fontsize=10, fontweight='bold')
    ax_hex.legend(fontsize=8)
    ax_hex.grid(alpha=0.3)
    ax_hex.text(0.05, 0.97,
                f'R²={r2_w:.3f}\nRMSE={rmse_w:.2f}\nBias={bias_w:+.2f}\nN={len(df_w):,}',
                transform=ax_hex.transAxes, ha='left', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.85))

fig.suptitle(f'{REF_METHOD} — Best Well vs Problematic Well\nSergipe-Alagoas Basin | LOWO',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'profile_best_vs_worst_{REF_METHOD.lower()}.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Well distribution by R² range (all methods)
ml_methods_bar = [m for m in ['XGBoost', 'HistGB', 'LightGBM', 'RandomForest', 'MLP', 'CNN']
                  if m in df_all['Method'].values]
bins_bar   = [-np.inf, 0.50, 0.70, 0.90, np.inf]
labels_bar = ['R² < 0.50', '0.50–0.70', '0.70–0.90', 'R² ≥ 0.90']

fig, ax = plt.subplots(figsize=(12, 6))
x     = np.arange(len(labels_bar))
width = 0.13
colors_bar = ['#E74C3C','#E67E22','#27AE60','#1A5C35','#2980B9','#8E44AD']

for i, method in enumerate(ml_methods_bar):
    df_m   = df_all[df_all['Method'] == method]
    counts = pd.cut(df_m['R²'], bins=bins_bar, labels=labels_bar).value_counts()
    pcts   = (counts / len(df_m) * 100).reindex(labels_bar).fillna(0)
    bars   = ax.bar(x + i * width, pcts.values, width,
                    label=method, color=colors_bar[i],
                    edgecolor='black', linewidth=0.8, alpha=0.85)
    for bar, val in zip(bars, pcts.values):
        if val > 3:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{val:.0f}%', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x + width * (len(ml_methods_bar) - 1) / 2)
ax.set_xticklabels(labels_bar, fontsize=11, fontweight='bold')
ax.set_ylabel('Percentage of Wells (%)', fontsize=12, fontweight='bold')
ax.set_title('Well Distribution by R² Range — LOWO\nSergipe-Alagoas Basin | 27 wells',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'distribution_r2_ranges_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()

# Numeric table
print('Table: % of wells by R² range')
print(f'{"Method":20s}', end='')
for label in labels_bar:
    print(f'  {label:12s}', end='')
print()
print('-' * 75)
for method in ml_methods_bar:
    df_m   = df_all[df_all['Method'] == method]
    counts = pd.cut(df_m['R²'], bins=bins_bar, labels=labels_bar).value_counts()
    pcts   = (counts / len(df_m) * 100).reindex(labels_bar).fillna(0)
    print(f'{method:20s}', end='')
    for val in pcts.values:
        print(f'  {val:5.1f}%       ', end='')
    print()


In [ ]:
# Computational time
if 'Time_seconds' in df_all.columns:
    time_df = (
        df_all[df_all['Time_seconds'].notna()]
        .groupby('Method')['Time_seconds']
        .agg(total_s='sum', mean_s='mean')
        .assign(total_min=lambda x: x['total_s'] / 60)
        .reset_index()
    )
    ml_order_time = ['XGBoost', 'LightGBM', 'HistGB', 'RandomForest', 'MLP', 'CNN']
    time_df = (time_df[time_df['Method'].isin(ml_order_time)]
               .set_index('Method')
               .reindex([m for m in ml_order_time if m in time_df['Method'].values])
               .reset_index())

    cat_colors_time = {
        'XGBoost':'steelblue','HistGB':'steelblue','LightGBM':'steelblue',
        'RandomForest':'steelblue','MLP':'coral','CNN':'coral'
    }
    colors_time = [cat_colors_time[m] for m in time_df['Method']]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    bars1 = ax1.barh(time_df['Method'], time_df['total_min'],
                     color=colors_time, edgecolor='black', linewidth=1.0)
    for bar, val in zip(bars1, time_df['total_min']):
        ax1.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f} min', va='center', fontsize=10, fontweight='bold')
    ax1.set_xlabel('Total LOWO Time [minutes]', fontsize=12, fontweight='bold')
    ax1.set_title('Total Time — Full LOWO\n(27 iterations)', fontsize=12, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)

    bars2 = ax2.barh(time_df['Method'], time_df['mean_s'],
                     color=colors_time, edgecolor='black', linewidth=1.0)
    for bar, val in zip(bars2, time_df['mean_s']):
        ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f} s', va='center', fontsize=10, fontweight='bold')
    ax2.set_xlabel('Mean Time per Well [seconds]', fontsize=12, fontweight='bold')
    ax2.set_title('Mean Time per LOWO Iteration', fontsize=12, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)

    legend_elements = [
        mpatches.Patch(facecolor='steelblue', edgecolor='black', label='Machine Learning'),
        mpatches.Patch(facecolor='coral',     edgecolor='black', label='Neural Network'),
    ]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.04))
    fig.suptitle('Computational Efficiency — LOWO | Sergipe-Alagoas Basin',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'computational_time_all_methods.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nCOMPUTATIONAL SUMMARY:')
    print(f'{"Method":20s} {"Total (min)":>12s} {"Per well (s)":>14s}')
    print('-' * 50)
    for _, row in time_df.iterrows():
        print(f'{row["Method"]:20s} {row["total_min"]:>12.1f} {row["mean_s"]:>14.1f}')
else:
    print('⚠️  Time_seconds not available — run LOWO notebooks with time recording enabled.')
